# Build 90-Day Daily LSTM Input

This notebook reads two permanent Delta tables:

- `insider_us_nms_windows_v2`
- `insider_us_nms_trx_alert_table`

It creates a fixed 90-day daily sequence for each employee-account-anchor-date sample.

The notebook stops after producing LSTM-ready input. It does not split, scale, train, or evaluate a model.


In [ ]:
import numpy as np
import pandas as pd

from pyspark.sql import functions as F
from pyspark.sql.window import Window

## 1. Configuration

In [ ]:
WINDOW_TABLE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdad1s1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_windows_v2"
)

TRX_TABLE_PATH = (
    "abfss://ml-artifact-insider-us@dsapdafazprdad1s1.dfs.core.windows.net/"
    "ins_us_nms/v1/output/insider_us_nms_trx_alert_table"
)

SEQUENCE_LENGTH = 90
BUSINESS_START_HOUR = 8
BUSINESS_END_HOUR = 18

# Set to an integer for development testing, or None for the full dataset.
DEVELOPMENT_SAMPLE_LIMIT = None

## 2. Read permanent tables

In [ ]:
windows_raw = spark.read.format("delta").load(WINDOW_TABLE_PATH)
trx_raw = spark.read.format("delta").load(TRX_TABLE_PATH)

print("Window table schema")
windows_raw.printSchema()

print("Transaction table schema")
trx_raw.printSchema()

## 3. Build the sample spine

One sample is defined by:

```text
login_id + acct_nbr + anchor_date
```

The sequence ends on the anchor date and starts 89 days earlier.


In [ ]:
required_window_columns = ["login_id", "acct_nbr", "date", "label"]

missing_window_columns = [
    c for c in required_window_columns
    if c not in windows_raw.columns
]

if missing_window_columns:
    raise ValueError(
        f"Missing required window columns: {missing_window_columns}"
    )

optional_window_columns = [
    c for c in [
        "lookback_window_start",
        "fraud_date",
        "label_split",
        "cv_fold"
    ]
    if c in windows_raw.columns
]

samples = (
    windows_raw
    .select(
        F.col("login_id").cast("string").alias("login_id"),
        F.col("acct_nbr").cast("string").alias("acct_nbr"),
        F.to_date("date").alias("anchor_date"),
        F.col("label").cast("int").alias("label"),
        *optional_window_columns
    )
    .dropDuplicates(["login_id", "acct_nbr", "anchor_date"])
    .withColumn("sequence_end_date", F.col("anchor_date"))
    .withColumn(
        "sequence_start_date",
        F.date_sub(F.col("anchor_date"), SEQUENCE_LENGTH - 1)
    )
    .withColumn(
        "sample_id",
        F.sha2(
            F.concat_ws(
                "||",
                F.col("login_id"),
                F.col("acct_nbr"),
                F.col("anchor_date").cast("string")
            ),
            256
        )
    )
)

if DEVELOPMENT_SAMPLE_LIMIT is not None:
    samples = samples.limit(DEVELOPMENT_SAMPLE_LIMIT)

display(samples.limit(10))
print("Number of samples:", samples.count())

## 4. Prepare transaction-level fields

In [ ]:
required_trx_columns = ["login_id", "acct_nbr", "date"]

missing_trx_columns = [
    c for c in required_trx_columns
    if c not in trx_raw.columns
]

if missing_trx_columns:
    raise ValueError(
        f"Missing required transaction columns: {missing_trx_columns}"
    )

trx = (
    trx_raw
    .withColumn("login_id", F.col("login_id").cast("string"))
    .withColumn("acct_nbr", F.col("acct_nbr").cast("string"))
    .withColumn("event_date", F.to_date("date"))
)

if "transaction_type" in trx.columns:
    trx = (
        trx
        .withColumn(
            "is_inquiry",
            F.when(
                F.col("transaction_type").cast("string") == "3",
                1
            ).otherwise(0)
        )
        .withColumn(
            "is_maintenance",
            F.when(
                F.col("transaction_type").cast("string") == "4",
                1
            ).otherwise(0)
        )
    )
else:
    trx = (
        trx
        .withColumn("is_inquiry", F.lit(0))
        .withColumn("is_maintenance", F.lit(0))
    )

if "transaction_hour" in trx.columns:
    trx = trx.withColumn(
        "is_after_hours",
        F.when(
            (F.col("transaction_hour") < BUSINESS_START_HOUR)
            | (F.col("transaction_hour") >= BUSINESS_END_HOUR),
            1
        ).otherwise(0)
    )
elif "transaction_datetime" in trx.columns:
    trx = trx.withColumn(
        "is_after_hours",
        F.when(
            (F.hour("transaction_datetime") < BUSINESS_START_HOUR)
            | (F.hour("transaction_datetime") >= BUSINESS_END_HOUR),
            1
        ).otherwise(0)
    )
else:
    trx = trx.withColumn("is_after_hours", F.lit(0))

## 5. Calculate transaction gaps

The gap is calculated within each employee-account-day.


In [ ]:
if "transaction_datetime" in trx.columns:
    transaction_window = (
        Window
        .partitionBy("login_id", "acct_nbr", "event_date")
        .orderBy("transaction_datetime")
    )

    trx = (
        trx
        .withColumn(
            "previous_transaction_datetime",
            F.lag("transaction_datetime").over(transaction_window)
        )
        .withColumn(
            "gap_minutes",
            (
                F.col("transaction_datetime").cast("long")
                - F.col("previous_transaction_datetime").cast("long")
            ) / F.lit(60.0)
        )
    )
else:
    trx = trx.withColumn(
        "gap_minutes",
        F.lit(None).cast("double")
    )

## 6. Aggregate transactions to employee-account-day

In [ ]:
daily_aggregations = [
    F.count("*").alias("daily_touches"),
    F.sum("is_inquiry").alias("daily_inquiries"),
    F.sum("is_maintenance").alias("daily_maintenances"),
    F.sum("is_after_hours").alias("daily_after_hours_touches"),
    F.avg("gap_minutes").alias("daily_avg_gap_minutes")
]

if "seq_len_in_time" in trx.columns:
    daily_aggregations.append(
        F.avg("seq_len_in_time").alias("daily_mean_seq_len")
    )
else:
    daily_aggregations.append(
        F.lit(0.0).alias("daily_mean_seq_len")
    )

if "non_mon_seq_ind" in trx.columns:
    daily_aggregations.append(
        F.sum(
            F.when(F.col("non_mon_seq_ind") == 1, 1).otherwise(0)
        ).alias("daily_non_mon_touches")
    )
else:
    daily_aggregations.append(
        F.lit(0).alias("daily_non_mon_touches")
    )

daily_touch = (
    trx
    .groupBy("login_id", "acct_nbr", "event_date")
    .agg(*daily_aggregations)
)

display(daily_touch.limit(10))

## 7. Create a complete 90-day calendar for each sample

In [ ]:
sample_calendar = (
    samples
    .withColumn(
        "calendar_date",
        F.explode(
            F.sequence(
                F.col("sequence_start_date"),
                F.col("sequence_end_date"),
                F.expr("INTERVAL 1 DAY")
            )
        )
    )
    .withColumn(
        "sequence_day",
        F.datediff(
            F.col("calendar_date"),
            F.col("sequence_start_date")
        )
    )
)

display(sample_calendar.limit(10))

## 8. Join daily behavior to the 90-day calendar

No-activity days are kept and filled with zero.


In [ ]:
daily_sequence = (
    sample_calendar.alias("s")
    .join(
        daily_touch.alias("d"),
        on=[
            F.col("s.login_id") == F.col("d.login_id"),
            F.col("s.acct_nbr") == F.col("d.acct_nbr"),
            F.col("s.calendar_date") == F.col("d.event_date")
        ],
        how="left"
    )
    .select(
        "s.sample_id",
        "s.login_id",
        "s.acct_nbr",
        "s.anchor_date",
        "s.sequence_start_date",
        "s.sequence_end_date",
        "s.calendar_date",
        "s.sequence_day",
        "s.label",
        *[
            F.col(f"s.{c}")
            for c in optional_window_columns
        ],
        "d.daily_touches",
        "d.daily_inquiries",
        "d.daily_maintenances",
        "d.daily_after_hours_touches",
        "d.daily_avg_gap_minutes",
        "d.daily_mean_seq_len",
        "d.daily_non_mon_touches"
    )
)

base_daily_features = [
    "daily_touches",
    "daily_inquiries",
    "daily_maintenances",
    "daily_after_hours_touches",
    "daily_avg_gap_minutes",
    "daily_mean_seq_len",
    "daily_non_mon_touches"
]

daily_sequence = daily_sequence.fillna(
    0,
    subset=base_daily_features
)

## 9. Add simple time-aware features

In [ ]:
daily_sequence = (
    daily_sequence
    .withColumn(
        "has_touch",
        F.when(F.col("daily_touches") > 0, 1).otherwise(0)
    )
    .withColumn(
        "is_weekend",
        F.when(F.dayofweek("calendar_date").isin(1, 7), 1).otherwise(0)
    )
)

history_window = (
    Window
    .partitionBy("sample_id")
    .orderBy("sequence_day")
    .rowsBetween(Window.unboundedPreceding, -1)
)

daily_sequence = (
    daily_sequence
    .withColumn(
        "previous_touch_day",
        F.max(
            F.when(
                F.col("has_touch") == 1,
                F.col("sequence_day")
            )
        ).over(history_window)
    )
    .withColumn(
        "days_since_previous_touch",
        F.when(
            F.col("previous_touch_day").isNull(),
            F.col("sequence_day") + 1
        ).otherwise(
            F.col("sequence_day") - F.col("previous_touch_day")
        )
    )
    .drop("previous_touch_day")
)

## 10. Validate sequence structure

In [ ]:
sequence_quality = (
    daily_sequence
    .groupBy("sample_id")
    .agg(
        F.count("*").alias("number_of_days"),
        F.min("sequence_day").alias("min_sequence_day"),
        F.max("sequence_day").alias("max_sequence_day"),
        F.countDistinct("label").alias("number_of_labels")
    )
)

invalid_sequence_count = (
    sequence_quality
    .filter(
        (F.col("number_of_days") != SEQUENCE_LENGTH)
        | (F.col("min_sequence_day") != 0)
        | (F.col("max_sequence_day") != SEQUENCE_LENGTH - 1)
        | (F.col("number_of_labels") != 1)
    )
    .count()
)

duplicate_sample_day_count = (
    daily_sequence
    .groupBy("sample_id", "sequence_day")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("Invalid sequence count:", invalid_sequence_count)
print("Duplicate sample-day count:", duplicate_sample_day_count)

if invalid_sequence_count != 0:
    raise ValueError("Some samples do not contain a valid 90-day sequence.")

if duplicate_sample_day_count != 0:
    raise ValueError("Duplicate sample-day rows were found.")

## 11. Final long-format LSTM input

In [ ]:
LSTM_DAILY_FEATURES = [
    "daily_touches",
    "daily_inquiries",
    "daily_maintenances",
    "daily_after_hours_touches",
    "daily_avg_gap_minutes",
    "daily_mean_seq_len",
    "daily_non_mon_touches",
    "has_touch",
    "days_since_previous_touch",
    "is_weekend"
]

lstm_input_long = (
    daily_sequence
    .select(
        "sample_id",
        "login_id",
        "acct_nbr",
        "anchor_date",
        "sequence_start_date",
        "sequence_end_date",
        "calendar_date",
        "sequence_day",
        "label",
        *optional_window_columns,
        *LSTM_DAILY_FEATURES
    )
    .orderBy("sample_id", "sequence_day")
)

number_of_samples = (
    lstm_input_long
    .select("sample_id")
    .distinct()
    .count()
)

number_of_rows = lstm_input_long.count()

print("Number of LSTM samples:", number_of_samples)
print("Number of daily rows:", number_of_rows)
print("Expected daily rows:", number_of_samples * SEQUENCE_LENGTH)

display(lstm_input_long.limit(100))

## 12. Create one array row per sample

This object is closer to the tensor format expected by an LSTM.


In [ ]:
daily_feature_vector = F.array(
    *[
        F.col(c).cast("double")
        for c in LSTM_DAILY_FEATURES
    ]
)

lstm_input_array = (
    lstm_input_long
    .withColumn(
        "daily_feature_vector",
        daily_feature_vector
    )
    .groupBy(
        "sample_id",
        "login_id",
        "acct_nbr",
        "anchor_date",
        "label"
    )
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    F.col("sequence_day"),
                    F.col("daily_feature_vector")
                )
            )
        ).alias("ordered_daily_rows")
    )
    .withColumn(
        "feature_sequence",
        F.transform(
            F.col("ordered_daily_rows"),
            lambda row: row["daily_feature_vector"]
        )
    )
    .drop("ordered_daily_rows")
)

display(lstm_input_array.limit(10))

## 13. Optional small NumPy tensor example

Only a small sample is converted to pandas to avoid driver-memory issues.


In [ ]:
TENSOR_EXAMPLE_SAMPLE_COUNT = 1000

example_sample_ids = (
    lstm_input_long
    .select("sample_id")
    .distinct()
    .limit(TENSOR_EXAMPLE_SAMPLE_COUNT)
)

example_pd = (
    lstm_input_long
    .join(example_sample_ids, on="sample_id", how="inner")
    .select(
        "sample_id",
        "sequence_day",
        "label",
        *LSTM_DAILY_FEATURES
    )
    .orderBy("sample_id", "sequence_day")
    .toPandas()
)

grouped_examples = list(
    example_pd.groupby("sample_id", sort=False)
)

X_example = np.stack([
    group[LSTM_DAILY_FEATURES].to_numpy(dtype=np.float32)
    for _, group in grouped_examples
])

y_example = np.array([
    group["label"].iloc[0]
    for _, group in grouped_examples
], dtype=np.float32)

print("X_example shape:", X_example.shape)
print("y_example shape:", y_example.shape)

assert X_example.shape[1] == SEQUENCE_LENGTH
assert X_example.shape[2] == len(LSTM_DAILY_FEATURES)

## Final objects

- `lstm_input_long`: one row per sample-day.
- `lstm_input_array`: one row per sample with a 90-day feature sequence.
- `X_example`: small NumPy tensor example.
- `y_example`: one label per example sample.

The notebook intentionally stops before splitting, scaling, and model training.
